### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics, and debugging.
- Transforming prompts, tool selection, and output formatting.
- Adding retries, fallbacks, and early termination logic.
- Applying rate limits, guardrails, and PII detection.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [2]:
from langchain.chat_models import init_chat_model
model=init_chat_model("groq:qwen/qwen3-32b")
model

d:\2026-courses\agenticai\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B4B8F72E30>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B4B8F73520>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
- Long-running conversations that exceed context windows.
- Multi-turn dialogues with extensive history.
- Applications where preserving full conversation context matters.

In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, model_call_limit
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver(), # Save the conversation history in memory
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("messages",10), # Trigger the summarization when 10 messages are added
            keep=("messages",4) # Keep the last 4 messages in the conversation
        )
    ]
)

d:\2026-courses\agenticai\venv\lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [4]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d4b27e09-d68b-4a85-9e65-fac3698623e1'), AIMessage(content='<think>\nOkay, let\'s see. The user asked, "What is 2+2?" Hmm, that\'s a basic arithmetic question. I need to make sure I understand it correctly. They might be testing if I know simple addition, or maybe they\'re a student just starting out.\n\nFirst, I should recall the basic addition facts. 2 plus 2 is a common question, often used in examples. In standard arithmetic, adding two and two gives four. But wait, could there be any context where the answer isn\'t four? Like in modular arithmetic or different number bases? For example, in base 3, 2+2 would be 11 (which is 4 in decimal). But the user didn\'t specify any base, so it\'s safe to assume base 10.\n\nAlso, sometimes people ask this question metaphorically, like in political science references, but the user is likely just asking for the mathematical answer. They mi

## Token Size


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(), # Save the conversation history in memory
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens",550), # Trigger the summarization when 550 tokens are added
            keep=("tokens",200), # Keep the last 200 tokens in the conversation
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [8]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~150 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='99768eb4-f58b-4cdc-aed5-36af9e0f0c61'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking to find hotels in Paris. Let me check the tools provided. There's a function called search_hotels that takes a city parameter. The description says it returns a long response to use more tokens. Since the user wants hotels in Paris, I need to call this function with the city set to Paris. I'll make sure the parameters are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'zkhyk6b69', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 109, 'prompt_tokens': 155, 'total_tokens': 264, 'completion_time': 0.158793342, 'completion_tokens_details': {'reasoning_tokens': 84}, 'prompt_time': 0.008169425, 'prompt_tokens_

## Fraction

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from pydantic_core.core_schema import model_field

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model=model,
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~82 tokens (0.0641%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='b95b88c2-75be-4b4c-980f-955d59b41a4d'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the tools provided. There's a function called search_hotels that requires a city parameter. Since the user specified Paris, I need to call that function with the city set to Paris. I'll make sure the JSON is correctly formatted with the city name in the parameters. No other parameters are needed, just the city. Let me double-check the required fields to ensure I didn't miss anything. Yep, only the city is required. Alright, time to structure the tool call properly.\n", 'tool_calls': [{'id': 'fhqt3y3cb', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 136, 'prompt_tokens': 147, 'total_tokens': 283, 'comp